In [1]:
import pandas as pd
import time
from datetime import datetime
import os           

In [2]:
start=time.perf_counter()

# Load TRDL Data

In [3]:
from DatabaseCreator import DatabaseCreator
import CJDH_local_settings

#Run Database Creator
if __name__ == "__main__":
    db_creator = DatabaseCreator(db_settings=CJDH_local_settings.local_settings['FPL_Points_Predictor'])
    fpl_engine = db_creator.get_engine_for("fpl_data_analysis")

In [4]:
#add this info to a settings dict?
current_gw = 11
current_season = 20262027

In [5]:
#fpl_picks_history:
#combine fpl_picks and fpl_picks2 into a single sql table in the database
#add fpl_picks_{current_season} and fpl_picks_2_{current_season} into a single sql table in the database

In [6]:
# Erling Haaland goals vs xg in all seasons
view = db_creator.run_sql(f"""
SELECT player_name_id, season, event, fixture, goals_scored, expected_goals, expected_assists, assists
                          FROM playergw_transformed
                          WHERE expected_goals IS NOT NULL
                            AND player_name_id = 'Erling Haaland'
                          ORDER BY season, event, fixture
""")
view.pivot_table(index=['season'], values=['goals_scored', 'expected_goals', 'expected_assists', 'assists'], aggfunc='sum',margins=True,dropna=False)

,assists,expected_assists,expected_goals,goals_scored
season,,,,
20222023.0,9.0,1.33591,17.4943,36.0
20232024.0,8.0,2.18000,29.5700,27.0
20242025.0,3.0,2.04000,21.9000,22.0
20252026.0,8.0,2.67000,25.5000,27.0
20262027.0,0.0,0.92000,8.0900,9.0
All,28.0,9.14591,102.5543,121.0


In [7]:
# Erling Haaland goals vs xg in all seasons
view = db_creator.run_sql(f"""
SELECT *
FROM predictions
""")
view.columns

Index(['player_name_id', 'element', 'season', 'event', 'team_name', 'team_elo',
       'opp_team_name', 'opp_team_elo', 'team_attack_elo',
       'opp_team_attack_elo', 'team_defence_elo', 'opp_team_defence_elo',
       'position', 'value', 'fixture_count', 'mins_pred', 'xpoints',
       'xpoints_mins', 'xpoints_goals', 'xpoints_assists',
       'xg_overperf_factor', 'xa_overperf_factor', 'xpoints_clean_sheets',
       'xpoints_shots_saved', 'xpoints_goals_conceded', 'xpoints_defcon',
       'xpoints_bonus', 'xgp90_pred', 'xap90_pred', 'xgcp90_pred',
       'in_any_team', 'in_my_team', 'latest_any_team', 'latest_my_team',
       'prediction_timestamp', 'prediction_gameweek'],
      dtype='str')

In [8]:
print(f"\nOptimizing team selection for GW{current_gw+1} to GW38...")

from FPL_Optimizer import FPLOptimizer

remaining_gw_predictions = db_creator.run_sql(f"""
    
    WITH predictions_status AS (
    SELECT *
    FROM predictions
    LEFT JOIN player_status ON predictions.element = player_status.id
    )
    SELECT
        web_name,
        status,
        now_cost/10 as value,
        chance_of_playing_this_round,
        event,
        position,
        team_name,
        opp_team_name,
        xpoints,
        ROUND(mins_pred) as mins_pred
    FROM predictions_status
    WHERE latest_my_team = 1
        AND event BETWEEN {current_gw+1} AND 38
""")

if __name__ == "__main__":
    custom_columns = {
        'player_name': 'web_name',
        'team_name': 'team_name',
        'position': 'position',
        'value': 'value',
        'xpoints': 'xpoints'
    }
    
    optimizer = FPLOptimizer(
        df=remaining_gw_predictions[remaining_gw_predictions['event']==current_gw+1],
        budget=999.9,
        column_mapping=custom_columns
    )
    optimizer.optimize()

    # Get results as DataFrames
    # squad = optimizer.get_squad_dataframe()
    # starting_xi = optimizer.get_starting_xi_dataframe()
    result = optimizer.calculate_bench_boost_xpoints()
    result['event'] = current_gw+1



Optimizing team selection for GW12 to GW38...
FPL TEAM OPTIMIZER (Linear Programming)

Budget for 15-Player Squad: £999.9m

Solving optimization problem...

OPTIMAL 15-PLAYER SQUAD
Starting XI Formation: 5-4-1
Total Squad Cost: £100.0m / £999.9m
Total Weighted Expected Points: 51.66
Remaining Budget: £899.9m

--------------------------------GOALKEEPERS (2)---------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        A.Becker                       Liverpool    £5.5     3.63 
 ✅        Martinez                       Chelsea      £4.9     3.96 

---------------------------------DEFENDERS (5)----------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 ✅        Silva                          Bournemouth  £5.1     4.67 
 ✅        Justin  

In [9]:
print(f"Team Selection Details - GW {current_gw+1}")
## latest_my_team and in_my_team differ because the former adjusts to freehits by setting the latest team to the previous gw
view = db_creator.run_sql(f"""
    
    WITH predictions_status AS (
    SELECT *
    FROM predictions
    LEFT JOIN player_status ON predictions.element = player_status.id
    )
    SELECT
        web_name,
        status,
        chance_of_playing_this_round,
        event,
        position,
        team_name,
        xpoints,
        CAST(ROUND(mins_pred) as INT) as mins_pred,
        CAST(ROUND(team_elo) as INT) as team_elo,
        CAST(ROUND(team_attack_elo) as INT) as team_attack_elo,
        CAST(ROUND(team_defence_elo) as INT) as team_defence_elo,
        opp_team_name,
        CAST(ROUND(opp_team_elo) as INT) as opp_team_elo,
        CAST(ROUND(opp_team_attack_elo) as INT) as opp_team_attack_elo,
        CAST(ROUND(opp_team_defence_elo) as INT) as opp_team_defence_elo,
        ROUND(xgp90_pred, 2) as xgp90_pred,
        ROUND(xg_overperf_factor, 2) as xg_overperf_factor,
        ROUND(xap90_pred, 2) as xap90_pred,
        ROUND(xa_overperf_factor, 2) as xa_overperf_factor,
        ROUND(xgcp90_pred, 2) as xgcp90_pred
        

    FROM predictions_status
    WHERE latest_my_team = 1
        AND event = {current_gw+1}
    ORDER BY xpoints DESC
""")
# print(f"Squad Points: {round(view['xpoints'].sum()+view['xpoints'][0],2)}") #including captain points
view#.head(15)

Team Selection Details - GW 12


,web_name,status,chance_of_playing_this_round,event,position,team_name,xpoints,mins_pred,team_elo,team_attack_elo,team_defence_elo,opp_team_name,opp_team_elo,opp_team_attack_elo,opp_team_defence_elo,xgp90_pred,xg_overperf_factor,xap90_pred,xa_overperf_factor,xgcp90_pred
0,Gibbs-White,a,100.0,12.0,MID,Nott'm Forest,5.464253,86,1020,994,1161,Chelsea,1069,1135,1015,0.30,1.00,0.35,1.88,1.38
1,Haaland,a,100.0,12.0,FWD,Man City,5.169730,79,1221,1283,1218,Arsenal,1277,1175,1375,0.66,1.16,0.19,2.79,1.52
2,Tavernier,a,NaN,12.0,MID,Bournemouth,5.017509,85,1123,1154,1208,Fulham,1042,934,1197,0.24,1.00,0.26,1.26,1.28
3,Rayan,a,NaN,12.0,MID,Bournemouth,4.673159,83,1123,1154,1208,Fulham,1042,934,1197,0.27,1.00,0.11,1.00,1.23
4,Silva,a,NaN,12.0,DEF,Bournemouth,4.668428,87,1123,1154,1208,Fulham,1042,934,1197,0.06,1.00,0.07,1.00,1.28
5,Justin,a,NaN,12.0,DEF,Leeds,4.665058,87,1049,1041,1206,Coventry City,994,1020,949,0.05,1.00,0.07,1.00,1.14
6,Vuskovic,a,NaN,12.0,DEF,Brighton,4.453808,88,1116,1218,1261,Newcastle,1053,1091,1043,0.07,1.00,0.05,1.00,1.28
7,Barnes,a,NaN,12.0,MID,Newcastle,4.250547,85,1053,1091,1043,Brighton,1116,1218,1261,0.23,1.17,0.12,1.00,1.44
8,Calafiori,a,100.0,12.0,DEF,Arsenal,4.219027,74,1277,1175,1375,Man City,1221,1283,1218,0.07,1.00,0.11,1.67,1.19
9,Martinez,a,100.0,12.0,GK,Chelsea,3.961131,90,1069,1135,1015,Nott'm Forest,1020,994,1161,NaN,1.00,NaN,1.00,1.23


In [10]:
print(f"Best players to transfer out for the next 5 gameweeks: GW{current_gw+1} AND {current_gw+5}")
view = db_creator.run_sql(f"""
    SELECT
        player_name_id,
        latest_my_team,
        position,
        team_name,
        ROUND(AVG(opp_team_elo),1) AS avg_opp_team_elo,
        ROUND(SUM(xpoints),2) AS total_xpoints,
        ROUND(AVG(mins_pred),1) AS avg_mins_pred,
        ROUND(AVG(xgp90_pred),3) AS avg_xgp90_pred,
        ROUND(AVG(xg_overperf_factor),3) AS avg_xg_overperf_factor,
        ROUND(AVG(xap90_pred),3) AS avg_xap90_pred,
        ROUND(AVG(xa_overperf_factor),3) AS avg_xa_overperf_factor,
        ROUND(AVG(xgcp90_pred),3) AS avg_xgcp90_pred
    FROM predictions
    WHERE latest_my_team = 1
      AND event BETWEEN {current_gw+1} AND {current_gw+5}
      -- AND position = 'DEF'
      -- AND team_name IN ('Arsenal','Man United','Nottm Forest','Brentford','Brighton','Chelsea')
    GROUP BY player_name_id, position, team_name, latest_my_team
    ORDER BY total_xpoints ASC
""")
view.head(50)

Best players to transfer out for the next 5 gameweeks: GW12 AND 16


,player_name_id,latest_my_team,position,team_name,avg_opp_team_elo,total_xpoints,avg_mins_pred,avg_xgp90_pred,avg_xg_overperf_factor,avg_xap90_pred,avg_xa_overperf_factor,avg_xgcp90_pred
0,Emersonn Correia da Silva,1,FWD,Ipswich Town,1066.2,11.18,47.6,0.323,1.000,0.045,1.000,1.364
1,Igor Thiago Nascimento Rodrigues,1,FWD,Brentford,1136.2,17.01,54.6,0.548,1.000,0.160,1.000,1.779
2,Emiliano Martínez Romero,1,GK,Chelsea,1103.0,18.13,90.0,NaN,1.000,NaN,1.000,1.514
3,Alisson Becker,1,GK,Liverpool,1042.9,18.26,90.0,NaN,1.000,NaN,1.000,1.300
4,Cody Gakpo,1,MID,Liverpool,1042.9,18.73,62.7,0.217,1.000,0.233,1.335,1.238
5,James Justin,1,DEF,Leeds,1083.2,20.79,86.5,0.048,1.000,0.059,1.000,1.404
6,Luka Vušković,1,DEF,Brighton,1049.5,21.15,87.6,0.069,1.000,0.045,1.000,1.373
7,António João Pereira de Albuquerque Tavares da...,1,DEF,Bournemouth,1089.8,21.56,87.3,0.060,1.000,0.063,1.000,1.461
8,Matheus Nunes,1,DEF,Man City,1098.1,21.89,72.6,0.043,1.000,0.214,2.204,1.212
9,Harvey Barnes,1,MID,Newcastle,1059.5,22.21,85.3,0.244,1.171,0.134,1.000,1.376


In [11]:
## Multi-week Optimizer

# from FPL_Optimizer import FPLOptimizer

# remaining_gw_predictions = db_creator.run_sql(f"""
    
#     WITH predictions_status AS (
#     SELECT *
#     FROM predictions
#     LEFT JOIN player_status ON predictions.element = player_status.id
#     )
#     SELECT
#         web_name,
#         status,
#         now_cost/10 as value,
#         chance_of_playing_this_round,
#         event,
#         position,
#         team_name,
#         opp_team_name,
#         xpoints,
#         ROUND(mins_pred) as mins_pred
#     FROM predictions_status
#     WHERE latest_my_team = 1
#         AND event BETWEEN {current_gw+1} AND 38
# """)

if __name__ == "__main__":
    custom_columns = {
        'player_name': 'web_name',
        'team_name': 'team_name',
        'position': 'position',
        'value': 'value',
        'xpoints': 'xpoints'
    }
    
    # 1. Initialize as an empty list to store DataFrames
    all_results_list = [] 

    for event in range(current_gw + 1, current_gw + 11):
        print(f"\nOptimizing team selection for GW{event}...")
        try:
            optimizer = FPLOptimizer(
                df=remaining_gw_predictions[remaining_gw_predictions['event'] == event],
                budget=102.4,
                column_mapping=custom_columns
            )
            optimizer.optimize()
        
            squad = optimizer.get_squad_dataframe()
            squad['event'] = event
            
            # 2. Append the individual DataFrame to the list
            all_results_list.append(squad) 

        except Exception as e:
            print(f"Error optimizing for GW{event}: {e}")

    # 3. Concatenate everything into one master DataFrame at the end
    if all_results_list:
        all_results = pd.concat(all_results_list, ignore_index=True)
        print("\nOptimization complete. Final DataFrame created.")
    else:
        all_results = pd.DataFrame() # Fallback if list is empty
        print("\nNo results found.")

all_results.pivot_table(index=['web_name','position'],columns='event',values='Starter',aggfunc='sum').reset_index().sort_values(by='position')


Optimizing team selection for GW12...
FPL TEAM OPTIMIZER (Linear Programming)

Budget for 15-Player Squad: £102.4m

Solving optimization problem...

OPTIMAL 15-PLAYER SQUAD
Starting XI Formation: 5-4-1
Total Squad Cost: £100.0m / £102.4m
Total Weighted Expected Points: 51.66
Remaining Budget: £2.4m

--------------------------------GOALKEEPERS (2)---------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        A.Becker                       Liverpool    £5.5     3.63 
 ✅        Martinez                       Chelsea      £4.9     3.96 

---------------------------------DEFENDERS (5)----------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 ✅        Silva                          Bournemouth  £5.1     4.67 
 ✅        Justin            

event,web_name,position,12,13,14,15,16,17,18,19,20,21
2,Calafiori,DEF,1,1,1,1,1,1,1,1,1,1
7,Justin,DEF,1,0,1,1,1,1,1,1,0,1
9,Matheus N.,DEF,1,1,1,1,1,1,1,1,1,1
11,Silva,DEF,1,1,1,1,1,1,1,0,1,1
14,Vuskovic,DEF,1,1,1,1,1,1,1,1,1,0
3,Emersonn,FWD,0,0,0,0,0,0,0,0,0,0
6,Haaland,FWD,1,1,1,1,1,1,1,1,1,1
13,Thiago,FWD,0,0,0,0,0,0,0,0,0,0
0,A.Becker,GK,0,0,1,1,0,0,0,1,0,0
8,Martinez,GK,1,1,0,0,1,1,1,0,1,1


In [12]:
print(f"Best players for next 5 gameweeks with exponential decay (normalized): GW{current_gw+1} to GW{current_gw+5}")

# For weights to sum to 1 with decay_factor d and initial weight w:
# w * (1 + d + d^2 + d^3 + d^4) = 1
# So w = 1 / (1 + d + d^2 + d^3 + d^4)

decay_factor = 0.5 #  lower decay factor means more emphasis on nearer gws, 1 is equal weights.
weights = [decay_factor**i for i in range(5)]
weight_sum = sum(weights)
initial_weight = 1 / weight_sum # weights add to 1
print(f"Decay factor: {decay_factor}, Weights: {[round((w/weight_sum),2) for w in weights]}")

view = db_creator.run_sql(f"""
    WITH 
    predictions_status AS (
        SELECT 
            p.player_name_id,
            p.element,
            ps.status,
            p.latest_my_team,
            p.position,
            p.team_name,
            p.opp_team_elo,
            p.xpoints,
            p.mins_pred,
            p.xgp90_pred,
            p.xg_overperf_factor,
            p.xap90_pred,
            p.xa_overperf_factor,
            p.xgcp90_pred,
            p.event,
            p.latest_any_team
        FROM predictions p
        LEFT JOIN player_status ps ON p.element = ps.id
    )
                                         
    SELECT
        player_name_id,
        status,
        latest_my_team,
        position,
        team_name,
        ROUND(SUM(
            CASE 
                WHEN event = {current_gw+1} THEN opp_team_elo * {initial_weight}
                WHEN event = {current_gw+2} THEN opp_team_elo * {initial_weight * decay_factor}
                WHEN event = {current_gw+3} THEN opp_team_elo * {initial_weight * decay_factor**2}
                WHEN event = {current_gw+4} THEN opp_team_elo * {initial_weight * decay_factor**3}
                WHEN event = {current_gw+5} THEN opp_team_elo * {initial_weight * decay_factor**4}
                ELSE 0
            END
        ), 1) AS avg_elo_decayed,
        ROUND(SUM(
            CASE 
                WHEN event = {current_gw+1} THEN xpoints * {initial_weight}
                WHEN event = {current_gw+2} THEN xpoints * {initial_weight * decay_factor}
                WHEN event = {current_gw+3} THEN xpoints * {initial_weight * decay_factor**2}
                WHEN event = {current_gw+4} THEN xpoints * {initial_weight * decay_factor**3}
                WHEN event = {current_gw+5} THEN xpoints * {initial_weight * decay_factor**4}
                ELSE 0
            END
        ), 2) AS total_xpoints_decayed,
        ROUND(AVG(mins_pred),1) AS avg_mins_pred,
        ROUND(AVG(xgp90_pred),3) AS avg_xgp90_pred,
        ROUND(AVG(xg_overperf_factor),3) AS avg_xg_overperf_factor,
        ROUND(AVG(xap90_pred),3) AS avg_xap90_pred,
        ROUND(AVG(xa_overperf_factor),3) AS avg_xa_overperf_factor,
        ROUND(AVG(xgcp90_pred),3) AS avg_xgcp90_pred
    FROM predictions_status
    WHERE (latest_any_team = 0 OR latest_my_team = 1)
      AND event BETWEEN {current_gw+1} AND {current_gw+5}
      AND status IN ('a','d')
      -- AND team_name IN ('Crystal Palace')
      -- AND position = 'FWD'
    GROUP BY player_name_id, status, position, team_name, latest_my_team
    ORDER BY total_xpoints_decayed DESC
""")
view.head(50)

Best players for next 5 gameweeks with exponential decay (normalized): GW12 to GW16
Decay factor: 0.5, Weights: [0.52, 0.26, 0.13, 0.06, 0.03]


,player_name_id,status,latest_my_team,position,team_name,avg_elo_decayed,total_xpoints_decayed,avg_mins_pred,avg_xgp90_pred,avg_xg_overperf_factor,avg_xap90_pred,avg_xa_overperf_factor,avg_xgcp90_pred
0,Erling Haaland,a,1,FWD,Man City,1170.3,5.53,78.1,0.836,1.156,0.203,2.792,1.448
1,Morgan Gibbs-White,a,1,MID,Nott'm Forest,1059.7,5.40,85.6,0.301,1.000,0.307,1.877,1.391
2,Marcus Tavernier,a,1,MID,Bournemouth,1071.9,4.96,84.7,0.249,1.000,0.254,1.260,1.398
3,Jack Rudoni,a,0,MID,Coventry City,1046.1,4.74,85.6,0.206,1.000,0.170,1.000,1.522
4,Rayan Vitor Simplício Rocha,a,1,MID,Bournemouth,1071.9,4.64,83.0,0.268,1.000,0.106,1.000,1.411
5,Noni Madueke,a,0,MID,Arsenal,1145.0,4.61,75.4,0.284,1.481,0.183,1.000,1.111
6,Mohammed Kudus,a,0,MID,Spurs,1068.3,4.51,84.4,0.161,1.000,0.245,1.837,1.536
7,Nobel Mendy,a,0,DEF,Hull City,1021.0,4.43,85.3,0.094,1.000,0.023,1.000,1.418
8,António João Pereira de Albuquerque Tavares da...,a,1,DEF,Bournemouth,1071.9,4.42,87.3,0.060,1.000,0.063,1.000,1.461
9,Riccardo Calafiori,a,1,DEF,Arsenal,1145.0,4.41,74.3,0.073,1.000,0.112,1.668,1.018


In [13]:
print(f"Best players to pick for the rest of the season: GW{current_gw+1} AND GW38")
view = db_creator.run_sql(f"""
WITH predictions_status AS (
    SELECT 
        p.player_name_id,
        p.element,
        ps.status,
        p.latest_my_team,
        p.position,
        p.team_name,
        p.opp_team_elo,
        p.xpoints,
        p.mins_pred,
        p.xgp90_pred,
        p.xg_overperf_factor,
        p.xap90_pred,
        p.xa_overperf_factor,
        p.xgcp90_pred,
        p.event,
        p.latest_any_team
    FROM predictions p
    LEFT JOIN player_status ps ON p.element = ps.id
)  
SELECT
    player_name_id,
    status,
    latest_my_team,
    position,
    team_name,
    ROUND(AVG(opp_team_elo),1) AS avg_opp_team_elo,
    ROUND(SUM(xpoints),2) AS total_xpoints,
    ROUND(AVG(mins_pred),1) AS avg_mins_pred,
    ROUND(AVG(xg_overperf_factor),3) AS avg_xg_overperf_factor,
    ROUND(AVG(xgp90_pred),3) AS avg_xgp90_pred,
    ROUND(AVG(xa_overperf_factor),3) AS avg_xa_overperf_factor,
    ROUND(AVG(xap90_pred),3) AS avg_xap90_pred,
    ROUND(AVG(xgcp90_pred),3) AS avg_xgcp90_pred
FROM predictions_status
WHERE (latest_any_team = 0 OR latest_my_team = 1)
    AND status IN ('a','d')
    -- AND position IN ('DEF')
    -- AND team_name = 'Liverpool'
  AND event BETWEEN {current_gw+1} AND 38
GROUP BY player_name_id, status, position, team_name, latest_my_team
ORDER BY total_xpoints DESC
""")
view.head(50)

Best players to pick for the rest of the season: GW12 AND GW38


,player_name_id,status,latest_my_team,position,team_name,avg_opp_team_elo,total_xpoints,avg_mins_pred,avg_xg_overperf_factor,avg_xgp90_pred,avg_xa_overperf_factor,avg_xap90_pred,avg_xgcp90_pred
0,Erling Haaland,a,1,FWD,Man City,1070.3,157.13,78.0,1.156,0.845,2.792,0.205,1.424
1,Morgan Gibbs-White,a,1,MID,Nott'm Forest,1072.8,144.64,85.6,1.000,0.300,1.877,0.315,1.398
2,Marcus Tavernier,a,1,MID,Bournemouth,1072.0,135.14,84.7,1.000,0.252,1.260,0.258,1.370
3,Noni Madueke,a,0,MID,Arsenal,1065.2,128.32,75.4,1.481,0.287,1.000,0.188,1.057
4,Jack Rudoni,a,0,MID,Coventry City,1081.3,126.33,85.5,1.000,0.204,1.000,0.167,1.576
5,Rayan Vitor Simplício Rocha,a,1,MID,Bournemouth,1072.0,126.02,83.0,1.000,0.270,1.000,0.109,1.355
6,Riccardo Calafiori,a,1,DEF,Arsenal,1065.2,124.64,74.2,1.000,0.073,1.668,0.117,0.981
7,Cristhian Mosquera,a,0,DEF,Arsenal,1065.2,123.63,59.1,1.000,0.121,1.000,0.043,1.089
8,Matheus Nunes,d,1,DEF,Man City,1070.3,121.44,72.6,1.000,0.043,2.204,0.220,1.153
9,Mohammed Kudus,a,0,MID,Spurs,1092.5,121.07,84.4,1.000,0.163,1.837,0.253,1.501


In [14]:
# print(f"Best available players to pick for the next 5 gameweeks: GW{current_gw+1} AND {current_gw+5}")
# view = db_creator.run_sql(f"""
# WITH predictions_status AS (
#     SELECT 
#         p.player_name_id,
#         p.element,
#         ps.status,
#         p.latest_my_team,
#         p.position,
#         p.team_name,
#         p.opp_team_elo,
#         p.xpoints,
#         p.mins_pred,
#         p.xgp90_pred,
#         p.xg_overperf_factor,
#         p.xap90_pred,
#         p.xa_overperf_factor,
#         p.xgcp90_pred,
#         p.event,
#         p.latest_any_team
#     FROM predictions p
#     LEFT JOIN player_status ps ON p.element = ps.id
# )  
# SELECT
#         player_name_id,
#         status,
#         latest_my_team,
#         position,
#         team_name,
#         ROUND(AVG(opp_team_elo),1) AS avg_opp_team_elo,
#         ROUND(SUM(xpoints),2) AS total_xpoints,
#         ROUND(AVG(mins_pred),1) AS avg_mins_pred,
#         ROUND(AVG(xgp90_pred),3) AS avg_xgp90_pred,
#         ROUND(AVG(xg_overperf_factor),3) AS avg_xg_overperf_factor,
#         ROUND(AVG(xap90_pred),3) AS avg_xap90_pred,
#         ROUND(AVG(xa_overperf_factor),3) AS avg_xa_overperf_factor,
#         ROUND(AVG(xgcp90_pred),3) AS avg_xgcp90_pred
#     FROM predictions_status
#     WHERE (latest_any_team = 0 OR latest_my_team = 1)
#       AND event BETWEEN {current_gw+1} AND {current_gw+5}
#       AND status IN ('a','d')
#         --AND position = 'FWD'
#       -- AND team_name IN ('Arsenal','Man United','Nottm Forest','Brentford','Brighton','Chelsea')
#     GROUP BY player_name_id, status, position, team_name, latest_my_team
#     ORDER BY total_xpoints DESC
# """)
# view.head(50)

In [15]:
print(f"\nBest available players to replace worst in my team: {current_gw+1} to {current_gw+5}")

view = db_creator.run_sql(f"""
    WITH 
    predictions_status AS (
        SELECT 
            p.player_name_id,
            p.element,
            ps.status,
            p.latest_my_team,
            p.position,
            p.team_name,
            p.opp_team_elo,
            p.xpoints,
            p.mins_pred,
            p.xgp90_pred,
            p.xap90_pred,
            p.xgcp90_pred,
            p.event,
            p.latest_any_team
        FROM predictions p
        LEFT JOIN player_status ps ON p.element = ps.id
    ),

    summed AS (
        SELECT 
            player_name_id,
            team_name,
                          status,
            position,
            latest_my_team,
            latest_any_team,
            SUM(xpoints) AS total_xpoints
        FROM predictions_status
        WHERE event BETWEEN {current_gw+1} AND {current_gw+5}
          AND status IN ('a', 'd')
        GROUP BY player_name_id, team_name, status, position, latest_my_team, latest_any_team
    ),

    worst_gk AS (
        SELECT player_name_id, team_name, status, total_xpoints AS xpoints, latest_my_team, position
        FROM summed
        WHERE latest_my_team = 1
          AND position = 'GK'
        ORDER BY total_xpoints ASC
        LIMIT 1
    ),
    best_gk AS (
        SELECT player_name_id, team_name, status, total_xpoints AS xpoints, latest_my_team, position
        FROM summed
        WHERE latest_any_team = 0
          AND latest_my_team = 0
          AND position = 'GK'
    ),

    worst_def AS (
        SELECT player_name_id, team_name, status, total_xpoints AS xpoints, latest_my_team, position
        FROM summed
        WHERE latest_my_team = 1
          AND position = 'DEF'
        ORDER BY total_xpoints ASC
        LIMIT 1
    ),
    best_def AS (
        SELECT player_name_id, team_name, status, total_xpoints AS xpoints, latest_my_team, position
        FROM summed
        WHERE latest_any_team = 0
          AND latest_my_team = 0
          AND position = 'DEF'
    ),

    worst_mid AS (
        SELECT player_name_id, team_name, status, total_xpoints AS xpoints, latest_my_team, position
        FROM summed
        WHERE latest_my_team = 1
          AND position = 'MID'
        ORDER BY total_xpoints ASC
        LIMIT 1
    ),
    best_mid AS (
        SELECT player_name_id, team_name, status, total_xpoints AS xpoints, latest_my_team, position
        FROM summed
        WHERE latest_any_team = 0
          AND latest_my_team = 0
          AND position = 'MID'
    ),

    worst_fwd AS (
        SELECT player_name_id, team_name, status, total_xpoints AS xpoints, latest_my_team, position
        FROM summed
        WHERE latest_my_team = 1
          AND position = 'FWD'
        ORDER BY total_xpoints ASC
        LIMIT 1
    ),
    best_fwd AS (
        SELECT player_name_id, team_name, status, total_xpoints AS xpoints, latest_my_team, position
        FROM summed
        WHERE latest_any_team = 0
          AND latest_my_team = 0
          AND position = 'FWD'
    )

    SELECT 
        b.position,
        b.team_name AS best_team,
        b.status AS status, 
        b.player_name_id AS best_player_id,
        b.xpoints AS best_xpoints,
        w.player_name_id AS worst_player_id,
        w.xpoints AS worst_xpoints,
        (b.xpoints - w.xpoints) AS xpoints_diff
    FROM best_gk b CROSS JOIN worst_gk w

    UNION ALL

    SELECT 
        b.position,
        b.team_name AS best_team,
        b.status AS status, 
        b.player_name_id AS best_player_id,
        b.xpoints AS best_xpoints,
        w.player_name_id AS worst_player_id,
        w.xpoints AS worst_xpoints,
        (b.xpoints - w.xpoints) AS xpoints_diff
    FROM best_def b CROSS JOIN worst_def w

    UNION ALL

    SELECT 
        b.position,
        b.team_name AS best_team,
        b.status AS status, 
        b.player_name_id AS best_player_id,
        b.xpoints AS best_xpoints,
        w.player_name_id AS worst_player_id,
        w.xpoints AS worst_xpoints,
        (b.xpoints - w.xpoints) AS xpoints_diff
    FROM best_mid b CROSS JOIN worst_mid w

    UNION ALL

    SELECT 
        b.position,
        b.team_name AS best_team,
        b.status AS status,
        b.player_name_id AS best_player_id,
        b.xpoints AS best_xpoints,
        w.player_name_id AS worst_player_id,
        w.xpoints AS worst_xpoints,
        (b.xpoints - w.xpoints) AS xpoints_diff
    FROM best_fwd b CROSS JOIN worst_fwd w
    ORDER BY xpoints_diff DESC
""")
view.head(20)


Best available players to replace worst in my team: 12 to 16


,position,best_team,status,best_player_id,best_xpoints,worst_player_id,worst_xpoints,xpoints_diff
0,MID,Coventry City,a,Jack Rudoni,23.524886,Cody Gakpo,18.731514,4.793371
1,MID,Arsenal,a,Noni Madueke,23.422326,Cody Gakpo,18.731514,4.690812
2,GK,Coventry City,a,Dan Bentley,22.099256,Emiliano Martínez Romero,18.133202,3.966054
3,GK,Coventry City,a,Ben Wilson,22.099256,Emiliano Martínez Romero,18.133202,3.966054
4,MID,Spurs,a,Mohammed Kudus,22.187075,Cody Gakpo,18.731514,3.455561
5,GK,Ipswich Town,a,David Button,21.561866,Emiliano Martínez Romero,18.133202,3.428664
6,FWD,Coventry City,d,Taiwo Awoniyi,14.503412,Emersonn Correia da Silva,11.183596,3.319816
7,FWD,Spurs,a,Dominic Solanke-Mitchell,14.431067,Emersonn Correia da Silva,11.183596,3.247471
8,GK,Nott'm Forest,a,Steven Benda,21.348113,Emiliano Martínez Romero,18.133202,3.214911
9,MID,Fulham,a,Alex Iwobi,21.639489,Cody Gakpo,18.731514,2.907975


In [16]:
player_name = 'Erling Haaland'
print(f"\n {player_name} xPoints between {current_gw+1} and {current_gw+5}:")
view = db_creator.run_sql(f"""
    SELECT
        player_name_id,
        event,
        position,
        team_name,
        opp_team_name,
        value/10 as value,
        opp_team_elo,
        xpoints,
        mins_pred,
        xgp90_pred,
        xap90_pred,
        xgcp90_pred
    FROM predictions
    WHERE event BETWEEN {current_gw+1} AND {current_gw+5}
        AND player_name_id = '{player_name}'
        -- AND value <= 48
    ORDER BY event ASC
""")
view.head(20)                                                              



 Erling Haaland xPoints between 12 and 16:


,player_name_id,event,position,team_name,opp_team_name,value,opp_team_elo,xpoints,mins_pred,xgp90_pred,xap90_pred,xgcp90_pred
0,Erling Haaland,12.0,FWD,Man City,Arsenal,15.8,1276.771766,5.169730,78.710623,0.657060,0.193837,1.523111
1,Erling Haaland,13.0,FWD,Man City,Leeds,15.8,1049.350902,5.943850,77.980623,0.884437,0.205616,1.452398
2,Erling Haaland,14.0,FWD,Man City,Brentford,15.8,1074.476464,5.813044,77.980623,0.845989,0.198930,1.404724
3,Erling Haaland,15.0,FWD,Man City,Chelsea,15.8,1068.679417,5.975328,77.870623,0.891810,0.203766,1.405161
4,Erling Haaland,16.0,FWD,Man City,Hull City,15.8,1021.161917,6.051409,77.980623,0.903156,0.212523,1.452687


# Chip Decisions

### Triple Captain

In [17]:
print("When to use TRIPLE CAPTAIN chip (gws1-18)")
view = db_creator.run_sql(f"""
    SELECT
        player_name_id,
        event,
        position,
        team_name,
        opp_team_name,
        xpoints
    FROM predictions
    WHERE in_my_team = 1
      AND event BETWEEN {current_gw} AND 18
    ORDER BY xpoints DESC
""")
view.head(10)

When to use TRIPLE CAPTAIN chip (gws1-18)


,player_name_id,event,position,team_name,opp_team_name,xpoints
0,Erling Haaland,16.0,FWD,Man City,Hull City,6.051409
1,Erling Haaland,17.0,FWD,Man City,Newcastle,6.027113
2,Erling Haaland,15.0,FWD,Man City,Chelsea,5.975328
3,Erling Haaland,13.0,FWD,Man City,Leeds,5.943850
4,Erling Haaland,18.0,FWD,Man City,Everton,5.893051
5,Erling Haaland,14.0,FWD,Man City,Brentford,5.813044
6,Morgan Gibbs-White,12.0,MID,Nott'm Forest,Chelsea,5.464253
7,Morgan Gibbs-White,17.0,MID,Nott'm Forest,Man Utd,5.451595
8,Morgan Gibbs-White,18.0,MID,Nott'm Forest,Newcastle,5.424434
9,Morgan Gibbs-White,15.0,MID,Nott'm Forest,Sunderland,5.404748


### Bench Boost

In [18]:
print(f"\nOptimizing team selection for GW{current_gw+1} to GW38...")

if __name__ == "__main__":
    custom_columns = {
        'player_name': 'web_name',
        'team_name': 'team_name',
        'position': 'position',
        'value': 'value',
        'xpoints': 'xpoints'
    }
    all_results =[]
    for event in range(current_gw+1, 39):
        print(f"\nOptimizing team selection for GW{event}...")
        try:
            optimizer = FPLOptimizer(
                df=remaining_gw_predictions[remaining_gw_predictions['event']==event],
                budget=999.9,
                column_mapping=custom_columns
            )
            optimizer.optimize()
        
            # Get results as DataFrames
            # squad = optimizer.get_squad_dataframe()
            # starting_xi = optimizer.get_starting_xi_dataframe()
            result = optimizer.calculate_bench_boost_xpoints()
            result['event'] = event
            # Append just the relevant keys as a dict
            all_results.append({
                'event': result['event'],
                'bench_boost_gain': result['bench_boost_gain']
            })
        except Exception as e:
            print(f"Error optimizing for GW{event}: {e}")



Optimizing team selection for GW12 to GW38...

Optimizing team selection for GW12...
FPL TEAM OPTIMIZER (Linear Programming)

Budget for 15-Player Squad: £999.9m

Solving optimization problem...

OPTIMAL 15-PLAYER SQUAD
Starting XI Formation: 5-4-1
Total Squad Cost: £100.0m / £999.9m
Total Weighted Expected Points: 51.66
Remaining Budget: £899.9m

--------------------------------GOALKEEPERS (2)---------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 🪑        A.Becker                       Liverpool    £5.5     3.63 
 ✅        Martinez                       Chelsea      £4.9     3.96 

---------------------------------DEFENDERS (5)----------------------------------
 Starter  Player Name                    Team         Cost     xPts 
--------------------------------------------------------------------------------
 ✅        Silva                          Bournem

In [19]:
print("Best weeks to Bench Boost:")
all_results_df = pd.DataFrame(all_results).sort_values(by='bench_boost_gain', ascending=False, inplace=False)
all_results_df

Best weeks to Bench Boost:


,event,bench_boost_gain
9,21,12.281301
7,19,11.933888
13,25,11.933888
19,31,11.789662
18,30,11.783624
14,26,11.744230
15,27,11.740508
4,16,11.740508
20,32,11.720503
5,17,11.716248


In [20]:
finish=time.perf_counter()
print(f'Finished in {round(((finish-start)/60),2)} minute(s)')
# Finished in 0.01 minute(s)

Finished in 0.1 minute(s)
